# 08 · Swiss roll: cierre de la isometría del latente

Corrida final de la sección del latente. Las preguntas y los criterios se fijaron **antes de correr**, a partir de lo que mostraron los notebooks 06 y 07:

| | Pregunta | Diseño | Criterio |
|---|---|---|---|
| **F1** | ¿El AE con continuación y `jac` recupera el rectángulo de forma fiable con costo aceptable? | `ae_warm`, λ ∈ {0, 0.03, 0.1}, **12 réplicas** | para cada λ > 0 en m = 64: Fisher unilateral contra λ = 0, p < 0.05, **y** mediana del costo ≤ 2× (con IC bootstrap al 95 %) |
| **F2** | ¿El costo es intrínseco a la clase afín de m modos? | mismas corridas | razón pareada MSE_x / MSE_x(oráculo → roll exacto), misma réplica y misma m: se reporta mediana e IC bootstrap. Si el IC contiene 1–1.5, el costo frente a λ = 0 es el precio de renunciar a la corrección no lineal del encoder, no un defecto del regularizador |
| **F3** | ¿En el VAE la KL compite con la isometría? | `vae_warm`, λ = 0.3, kl ∈ {10⁻³, 10⁻⁴}, 6 réplicas | con kl = 10⁻⁴ el MSE_x baja (Mann-Whitney unilateral p < 0.05) sin bajar la tasa de rectángulo |

Rectángulo v2 (igual que en el 07): desplegado (kNN ≤ 10 × piso) ∧ distorsión ≤ 2 × oráculo(M) ∧ R²_s ≥ 0.90 ∧ R²_h ≥ 0.80. Costo = MSE_x / mediana de MSE_x del mismo modelo con λ = 0.

Solo se entrena desde cero el modelo de m = 2 (inicio de cadena). Si adjuntas como *Input* la salida del 07, se reutilizan `vae` con λ = 0 y λ = 0.3, kl = 10⁻³ (mismas semillas 0–5). Duración ~3 h en dos T4.

## Preparación

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
ROOT = Path('/kaggle/working/dmap_vae_kaggle')
INPUT = Path('/kaggle/input')
CU = 'flash_diffusion.cu'
CODE = ['spectral.py', 'vae.py', 'data.py', 'experiment.py', 'launch.py', 'plots.py',
        'checks.py', 'prepare_alanine.py', 'generate_angles.py', 'SOURCE.json',
        'geometry.py', 'beta_sweep.py', 'protocol.py', 'config_beta_sweep.json',
        'config_beta_sweep_torus.json', 'config_beta_sweep_mnist.json', 'config_beta_sweep_alanine.json', 'latent_figure.py']

# 1) Refresh the code on EVERY run (a partial copy left in /kaggle/working is never trusted).
#    The results in /kaggle/working/results_* are not touched.
sources = sorted({q.parent for q in INPUT.rglob('launch.py') if (q.parent / 'spectral.py').exists()})
archives = sorted(INPUT.rglob('dmap_vae_kaggle*.zip'))
if len(sources) > 1:
    raise FileNotFoundError(f'Hay varias copias del paquete en /kaggle/input: {sources}. Deja solo una.')
ROOT.mkdir(parents=True, exist_ok=True)
if sources:
    shutil.copytree(sources[0], ROOT, dirs_exist_ok=True)
elif len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(ROOT.parent)
else:
    raise FileNotFoundError('Añade el paquete dmap_vae_kaggle como Kaggle Dataset; debe haber una sola copia.')
missing_code = [f for f in CODE if not (ROOT / f).exists()]
assert not missing_code, (f'Faltan archivos del paquete: {missing_code}. Este notebook necesita el '
                          'paquete v5 (dmap_vae_kaggle_v5.zip): sustituye el Dataset anterior.')

# 2) The CUDA source. Kaggle drops or flattens subfolders when a Dataset is built from loose
#    files, so look for it in vendor/, flattened at the root, or inside any attached ZIP.
target = ROOT / 'vendor' / CU
target.parent.mkdir(exist_ok=True)
if not target.exists():
    found = sorted(INPUT.rglob(CU))
    if found:
        shutil.copy2(found[0], target)
    else:
        for arc in sorted(INPUT.rglob('*.zip')):
            with zipfile.ZipFile(arc) as z:
                names = [n for n in z.namelist() if n.endswith('/' + CU) or n == CU]
                if names:
                    target.write_bytes(z.read(names[0]))
                    break
    lic = sorted(INPUT.rglob('LICENSE.FlashDiffusion'))
    if lic and not (target.parent / lic[0].name).exists():
        shutil.copy2(lic[0], target.parent / lic[0].name)
if not target.exists():
    raise FileNotFoundError(
        f'No se encontró {CU} en /kaggle/input. Sube la carpeta vendor/ (o el ZIP completo) '
        'al Kaggle Dataset; al subir archivos sueltos Kaggle omite las subcarpetas.')
expected = json.loads((ROOT / 'SOURCE.json').read_text())['sha256']
actual = hashlib.sha256(target.read_bytes()).hexdigest()
assert actual == expected, f'{CU} no coincide con SOURCE.json ({actual} != {expected}).'
print('Fuente CUDA verificada:', target)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import importlib.util
missing = [m for m in ['numpy','scipy','pandas','matplotlib','ninja']
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
assert torch.cuda.device_count() >= 2, 'Selecciona GPU T4 x2 en Kaggle.'
print([(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i)) for i in range(2)])
(ROOT / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))


## Controles del paquete (una GPU, ~3 min)

Compila FlashDiffusion y ejecuta `checks.py`: kernel contra matriz densa, convención de signo, métrica kNN, continuación con ceros, paso AE→VAE y los controles nuevos de v5 (regla de degeneración del toro, regla argmin, sonda de etiquetas de MNIST, rotación de trayectorias de alanina). Si falla, no sigas.

In [ ]:
import time
t0 = time.time()
r = subprocess.run([sys.executable, str(ROOT / 'checks.py'), '--backend', 'flash', '--device', 'cuda:0',
                    '--out', str(ROOT / 'checks_08.json')],
                   env=dict(os.environ, CUDA_VISIBLE_DEVICES='0'), capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:]); print(r.stderr[-6000:])
    raise RuntimeError('No continuar: falló un control del paquete.')
checks = json.loads((ROOT / 'checks_08.json').read_text())
print({k: v for k, v in checks.items() if isinstance(v, str)})
print(f"signo: {checks['sign_overlap_min']:.6f} | kNN suave/plegada: {checks['knn_smooth_ratio']:.2f} / "
      f"{checks['knn_folded_ratio']:.0f} veces el piso | {(time.time()-t0)/60:.1f} min")

## β y resultados previos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
FACTOR = 64
for prev in [Path('/kaggle/working/beta_sweep_swiss/beta_selection.json')] + sorted(Path('/kaggle/input').rglob('beta_sweep_swiss/beta_selection.json')):
    if prev.exists():
        FACTOR = json.loads(prev.read_text())['selected_factor']
        break
print('factor beta =', FACTOR)

def previo(nombre):
    for base in [Path('/kaggle/working')] + [p.parent for p in sorted(Path('/kaggle/input').rglob(nombre))]:
        if (base / nombre / 'metrics.csv').exists():
            return base / nombre
    return None
PREV07 = {0.0: previo('results_swiss_lambda_jac_l0') or previo('results_swiss_latente_R1_sin_reg'),
          0.3: previo('results_swiss_lambda_jac_l0.3')}
print({k: str(v) if v else 'se correrá aquí' for k, v in PREV07.items()})

## Plan y estimación de tiempo

El orden es la prioridad: primero F1 (AE, 12 réplicas) y después F3 (VAE). λ = 0 del AE se corre aquí con las 12 réplicas, porque el costo y Fisher necesitan el mismo número de réplicas.

In [ ]:
import math
import protocol as P
BASE = json.loads((ROOT / 'config_swiss.json').read_text())
BASE.update(width=60., height=10.)
COMMON = dict(P.COMMON)
LADDER = [2, 4, 8, 16, 64]
M = max(LADDER)
SEEDS_AE = list(range(12))
SEEDS_VAE = list(range(6))
comun = dict(beta_factor=FACTOR, modes=LADDER, m_protocols=['scratch', 'warm'], mode_order='spectral',
             oracle_decoder=False, reg_k=10, scratch_first_only=True)
PLAN = []
AE = {}
for lam in (0.03, 0.1, 0.0):
    tag = f'AE_l{lam:g}'
    p = dict(tag=tag, models=['ae'], seeds=SEEDS_AE, out=f'/kaggle/working/results_swiss_final_{tag}', **comun)
    if lam > 0:
        p.update(reg='jac', reg_weight=lam)
    PLAN.append(p); AE[lam] = Path(p['out'])
VAE = {}
for lam, kl in ((0.3, 1e-4), (0.3, 1e-3), (0.0, 1e-3)):
    if kl == 1e-3 and PREV07.get(lam) is not None:
        VAE[(lam, kl)] = PREV07[lam]; continue
    tag = f'VAE_l{lam:g}_kl{kl:g}'
    p = dict(tag=tag, models=['vae'], seeds=SEEDS_VAE, kl=kl, out=f'/kaggle/working/results_swiss_final_{tag}', **comun)
    if lam > 0:
        p.update(reg='jac', reg_weight=lam)
    PLAN.append(p); VAE[(lam, kl)] = Path(p['out'])
SECONDS_PER_MODEL = 80 * 1.4
P.show_plan(PLAN, 'swiss', SECONDS_PER_MODEL, COMMON)

## Ejecutar

Se puede re-ejecutar: las configuraciones completas se omiten y las parciales se señalan (bórralas o cambia el `tag`). Para una corrida larga conviene **Save Version → Save & Run All (Commit)**.

In [ ]:
import protocol as P
hechas, problemas = P.run_plan(PLAN, BASE, ROOT, common=COMMON)

## Análisis

Los intervalos del costo son bootstrap (5000 remuestreos de réplicas dentro de cada grupo, razón de medianas). La razón frente al oráculo es **pareada**: misma réplica, mismos datos y misma m.

In [ ]:
from scipy.stats import fisher_exact, mannwhitneyu
import protocol as P
from latent_figure import chart_r2_table
R2S, R2H = .90, .80
rng = np.random.default_rng(0)

def evalua(folder, model):
    d = P.load(folder)
    if d is None:
        return None
    q = P.with_chain_start(d)
    q = q[q.model == model].copy()
    ora = d[d.model == 'oracle'].set_index(['seed', 'm'])
    ref = d[(d.model == 'oracle') & (d.m == M)].set_index('seed').local_distortion
    q = q.merge(chart_r2_table(folder, [model]), on=['seed', 'model', 'm'], how='left')
    q['rect'] = (q.unfolded.astype(bool) & (q.local_distortion <= 2*q.seed.map(ref))
                 & (q.r2_s >= R2S) & (q.r2_h >= R2H))
    q['floor_x'] = [ora.oracle_exact_mse_x.get((s, m), np.nan) for s, m in zip(q.seed, q.m)]
    return q

def boot_ratio(a, b, n=5000):
    a, b = np.asarray(a), np.asarray(b)
    r = [np.median(rng.choice(a, len(a))) / np.median(rng.choice(b, len(b))) for _ in range(n)]
    return np.median(a)/np.median(b), *np.quantile(r, [.025, .975])

def boot_median(x, n=5000):
    x = np.asarray(x)
    return np.median(x), *np.quantile([np.median(rng.choice(x, len(x))) for _ in range(n)], [.025, .975])

def resumen(Q, base, etiqueta):
    filas = []
    for key, q in Q.items():
        if q is None:
            continue
        for m, g in q.groupby('m'):
            b = base[base.m == m].mse_x if base is not None else None
            k, n = int(g.rect.sum()), len(g)
            lo, hi = P.clopper_pearson(k, n)
            c, clo, chi = boot_ratio(g.mse_x, b) if b is not None and len(b) else (np.nan,)*3
            f, flo, fhi = boot_median(g.mse_x / g.floor_x)
            filas.append(dict(grupo=etiqueta, config=key, m=int(m), rectangulo=f'{k}/{n}', tasa=k/n, ic95_inf=lo, ic95_sup=hi,
                              r2_h=g.r2_h.median(), distorsion=g.local_distortion.median(), mse_x=g.mse_x.median(),
                              costo=c, costo_ic=f'{clo:.2f}–{chi:.2f}', vs_oraculo=f, vs_oraculo_ic=f'{flo:.2f}–{fhi:.2f}'))
    return pd.DataFrame(filas)

fmt = {'tasa': '{:.0%}', 'ic95_inf': '{:.2f}', 'ic95_sup': '{:.2f}', 'r2_h': '{:.3f}', 'distorsion': '{:.3g}',
       'mse_x': '{:.4g}', 'costo': '×{:.2f}', 'vs_oraculo': '×{:.2f}'}

# ---- F1 / F2: AE ----
QA = {lam: evalua(f, 'ae_warm') for lam, f in AE.items()}
TA = resumen(QA, QA.get(0.0), 'AE')
display(TA[TA.m == M].style.format(fmt).set_caption(f'F1/F2 · AE con continuación, m = {M}'))
for lam in (0.03, 0.1):
    a, b = QA.get(lam), QA.get(0.0)
    if a is None or b is None:
        continue
    a, b = a[a.m == M], b[b.m == M]
    ka, kb = int(a.rect.sum()), int(b.rect.sum())
    p = fisher_exact([[ka, len(a)-ka], [kb, len(b)-kb]], alternative='greater')[1]
    c, clo, chi = boot_ratio(a.mse_x, b.mse_x)
    ok = p < .05 and c <= 2
    print(f"F1 λ={lam:g}: rectángulo {ka}/{len(a)} vs {kb}/{len(b)}, Fisher p = {p:.3g}; costo ×{c:.2f} "
          f"(IC {clo:.2f}–{chi:.2f}) -> {'CUMPLE' if ok else 'no cumple'}")
    f, flo, fhi = boot_median(a.mse_x / a.floor_x)
    print(f"F2 λ={lam:g}: MSE_x / MSE_x(oráculo -> roll exacto) = ×{f:.2f} (IC {flo:.2f}–{fhi:.2f}); "
          f"sin regularizar: ×{np.median(b.mse_x/b.floor_x):.2f}")

# ---- F3: VAE, conflicto KL / isometría ----
QV = {k: evalua(f, 'vae_warm') for k, f in VAE.items()}
TV = resumen({f'λ={l:g}, kl={k:g}': q for (l, k), q in QV.items()}, QV.get((0.0, 1e-3)), 'VAE')
display(TV[TV.m == M].style.format(fmt).set_caption(f'F3 · VAE con continuación, m = {M}'))
a, b = QV.get((0.3, 1e-4)), QV.get((0.3, 1e-3))
if a is not None and b is not None:
    a, b = a[a.m == M], b[b.m == M]
    p = mannwhitneyu(a.mse_x, b.mse_x, alternative='less').pvalue
    print(f"F3: λ=0.3 kl=1e-4 MSE_x {a.mse_x.median():.4g} vs kl=1e-3 {b.mse_x.median():.4g}, Mann-Whitney p = {p:.3g}; "
          f"rectángulo {int(a.rect.sum())}/{len(a)} vs {int(b.rect.sum())}/{len(b)} -> "
          f"{'CUMPLE' if p < .05 and a.rect.mean() >= b.rect.mean() else 'no cumple'}")
pd.concat([TA, TV]).to_csv('/kaggle/working/final_tablas.csv', index=False)

In [ ]:
# ---- Figuras para el artículo ----
from latent_figure import latent_grid
rows = [dict(label='DMAP (lectura afín)', folder=AE[0.0], model='oracle'),
        dict(label='AE continuación', folder=AE[0.0], model='ae_warm'),
        dict(label='AE continuación\n+ jac λ=0.03', folder=AE[0.03], model='ae_warm'),
        dict(label='AE continuación\n+ jac λ=0.1', folder=AE[0.1], model='ae_warm')]
latent_grid(rows, LADDER, '/kaggle/working/final_latente',
            'Swiss roll 60×10: latente 2-D alineado afínmente a la carta (test, réplica mediana)')
plt.show()

fig, (a, b) = plt.subplots(1, 2, figsize=(12.5, 4.4))
estilo = {0.0: ('C7', 'o', 'AE continuación'), 0.03: ('C2', 's', 'AE + jac λ=0.03'), 0.1: ('C4', 'D', 'AE + jac λ=0.1')}
for i, (lam, (c, mk, lab)) in enumerate(estilo.items()):
    q = QA.get(lam)
    if q is None:
        continue
    s = q.groupby('m').mse_x
    a.plot(s.median().index, s.median().values, color=c, marker=mk, label=lab)
    a.fill_between(s.median().index, s.quantile(.25).values, s.quantile(.75).values, color=c, alpha=.15, lw=0)
    t = TA[TA.config == lam].sort_values('m')
    b.errorbar(t.m*1.04**(i-1), t.tasa, yerr=[t.tasa-t.ic95_inf, t.ic95_sup-t.tasa], color=c, marker=mk, capsize=3, label=lab)
fl = QA[0.0].groupby('m').floor_x.median()
a.plot(fl.index, fl.values, 'k--', lw=1, label='oráculo afín → roll exacto (piso de la clase)')
a.set(xscale='log', yscale='log', xlabel='m', ylabel='test MSE$_x$ (mediana, IQR)', title='Reconstrucción y piso de la clase afín')
b.set(xscale='log', ylim=(-.05, 1.05), xlabel='m', ylabel='fracción «rectángulo»', title='Rectángulo v2 (IC 95 %, 12 réplicas)')
for ax in (a, b):
    ax.set_xticks(LADDER); ax.set_xticklabels([str(m) for m in LADDER]); ax.minorticks_off(); ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig('/kaggle/working/final_frontera.png', dpi=200); fig.savefig('/kaggle/working/final_frontera.pdf')
plt.show()

## Qué descargar

`/kaggle/working/results_swiss_final_*`, `final_latente.png/.pdf`, `final_frontera.png/.pdf` y `final_tablas.csv`.